# 06 - Multi-Dataset Benchmarking (Tabular)

This notebook benchmarks all tabular noise detection methods across multiple datasets:
- IQR
- Z-Score
- DBSCAN
- Isolation Forest (Model-Based)

Supported datasets in this benchmark:
1. Pima Indians Diabetes
2. Chronic Kidney Disease
3. Wisconsin Breast Cancer
4. Stroke Prediction
5. Credit Card Fraud

## How To Use
1. Put all tabular datasets in the project root folder: `data/tabular/`.
2. Use Excel files (`.xlsx` or `.xls`) and set the exact file names in the configuration cell.
3. For labeled anomaly datasets, provide `label_column` and `anomaly_label`.
4. For unlabeled datasets, keep `label_column=None`; synthetic anomalies will be injected automatically for evaluation.
5. Run all cells and review the final comparison tables.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.detectors import (
    detect_outliers_iqr,
    detect_outliers_zscore,
    detect_noise_dbscan,
    detect_noise_isolation_forest,
)

In [ ]:

repo_root = project_root.parent
data_dir = repo_root / 'data' / 'Tabular'

DATASET_CONFIG = [
    {
        'name': 'Pima Indians Diabetes',
        'path': data_dir / 'Pima_Indians_Diabetes.csv',
        'label_column': None,
        'anomaly_label': 1,
    },
    {
        'name': 'Chronic Kidney Disease',
        'path': data_dir / 'kidney_disease.csv',
        'label_column': None,
        'anomaly_label': 1,
    },
    {
        'name': 'Wisconsin Breast Cancer',
        'path': data_dir / 'breast-cancer-wisconsin.csv',
        'label_column': None,
        'anomaly_label': 1,
    },
    {
        'name': 'Stroke Prediction',
        'path': data_dir / 'healthcare-dataset-stroke-data.csv',
        'label_column': 'stroke',
        'anomaly_label': 1,
    },
    {
        'name': 'Credit Card Fraud',
        'path': data_dir / 'creditcard_fraud_detection.csv',
        'label_column': 'Class',
        'anomaly_label': 1,
    },
]

DATASET_CONFIG

In [ ]:
def prepare_features(df, label_column=None):
    work = df.copy()
    y = None

    if label_column is not None and label_column in work.columns:
        y = work[label_column].copy()
        work = work.drop(columns=[label_column])

    
    for col in work.columns:
        if work[col].dtype == 'object':
            work[col] = work[col].replace('?', np.nan)

    numeric_cols = work.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in work.columns if c not in numeric_cols]

    for col in numeric_cols:
        work[col] = pd.to_numeric(work[col], errors='coerce')
        work[col] = work[col].fillna(work[col].median())

    for col in categorical_cols:
        mode = work[col].mode(dropna=True)
        fill_value = mode.iloc[0] if not mode.empty else 'missing'
        work[col] = work[col].fillna(fill_value).astype(str)

    encoded = pd.get_dummies(work, drop_first=False)

    scaler = StandardScaler()
    scaled = pd.DataFrame(scaler.fit_transform(encoded), columns=encoded.columns, index=encoded.index)

    numeric_only = work.select_dtypes(include=[np.number])
    return encoded, scaled, numeric_only, y


def row_flags_from_column_outliers(n_rows, column_to_indices):
    flags = np.zeros(n_rows, dtype=int)
    for idx_list in column_to_indices.values():
        flags[idx_list] = 1
    return flags


def run_iqr(numeric_df, multiplier=1.5):
    col_outliers = {}
    for col in numeric_df.columns:
        idx, _ = detect_outliers_iqr(numeric_df[col].values, multiplier=multiplier)
        col_outliers[col] = idx
    return row_flags_from_column_outliers(len(numeric_df), col_outliers)


def run_zscore(numeric_df, threshold=3.0):
    col_outliers = {}
    for col in numeric_df.columns:
        idx, _ = detect_outliers_zscore(numeric_df[col].values, threshold=threshold)
        col_outliers[col] = idx
    return row_flags_from_column_outliers(len(numeric_df), col_outliers)


def run_dbscan(encoded_df, eps=1.2, min_samples=10):
    labels = detect_noise_dbscan(encoded_df, eps=eps, min_samples=min_samples, scale=True)
    return (labels == -1).astype(int)


def run_isolation_forest(encoded_df, contamination=0.05):
    labels, scores = detect_noise_isolation_forest(encoded_df, contamination=contamination)
    flags = (labels == -1).astype(int)
    anomaly_score = -scores
    return flags, anomaly_score

In [ ]:
def inject_synthetic_anomalies(encoded_df, fraction=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    
    
    injected = encoded_df.astype(np.float64).copy()

    n = len(injected)
    k = max(1, int(n * fraction))
    anomaly_idx = rng.choice(n, size=k, replace=False)

    std = injected.std(axis=0).replace(0, 1.0)
    noise_matrix = rng.normal(loc=0.0, scale=6.0, size=(k, injected.shape[1]))
    base_values = injected.iloc[anomaly_idx, :].to_numpy(dtype=np.float64, copy=True)
    injected.iloc[anomaly_idx, :] = base_values + noise_matrix * std.to_numpy(dtype=np.float64)

    y_injected = np.zeros(n, dtype=int)
    y_injected[anomaly_idx] = 1
    return injected, y_injected


def evaluate_binary(y_true, y_pred, score=None):
    result = {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }

    if score is None:
        score = y_pred

    try:
        result['pr_auc'] = average_precision_score(y_true, score)
    except ValueError:
        result['pr_auc'] = np.nan

    return result

In [ ]:
all_results = []

for cfg in DATASET_CONFIG:
    dataset_name = cfg['name']
    path = Path(cfg['path'])

    if not path.exists():
        print(f'[SKIP] {dataset_name}: file not found -> {path}')
        continue

    print(f'[RUN] {dataset_name}')
    suffix = path.suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        df = pd.read_excel(path)
    elif suffix == '.csv':
        df = pd.read_csv(path)
    else:
        print(f'[SKIP] {dataset_name}: unsupported file type -> {suffix}')
        continue

    encoded, scaled, numeric_df, y = prepare_features(df, cfg['label_column'])

    if numeric_df.shape[1] == 0:
        print(f'[SKIP] {dataset_name}: no numeric columns for IQR/Z-Score')
        continue

    if y is not None:
        y_true = (y == cfg['anomaly_label']).astype(int).to_numpy()
        eval_mode = 'labeled'
        eval_encoded = encoded
    else:
        eval_encoded, y_true = inject_synthetic_anomalies(encoded, fraction=0.05, random_state=42)
        eval_mode = 'synthetic_injection'

    
    if y is None:
        eval_numeric = pd.DataFrame(
            StandardScaler().fit_transform(eval_encoded),
            columns=eval_encoded.columns,
            index=eval_encoded.index,
        )
    else:
        eval_numeric = numeric_df.reset_index(drop=True)

    iqr_pred = run_iqr(eval_numeric)
    z_pred = run_zscore(eval_numeric)
    db_pred = run_dbscan(eval_encoded, eps=1.2, min_samples=10)
    if_pred, if_score = run_isolation_forest(eval_encoded, contamination=0.05)

    method_outputs = [
        ('IQR', iqr_pred, None),
        ('Z-Score', z_pred, None),
        ('DBSCAN', db_pred, None),
        ('Isolation Forest', if_pred, if_score),
    ]

    for method_name, pred, score in method_outputs:
        metrics = evaluate_binary(y_true, pred, score=score)
        all_results.append({
            'dataset': dataset_name,
            'method': method_name,
            'mode': eval_mode,
            'samples': int(len(y_true)),
            'detected_anomalies': int(np.sum(pred)),
            **metrics,
        })

results_df = pd.DataFrame(all_results)
results_df

In [ ]:
if results_df.empty:
    print('No datasets were run. Check DATASET_CONFIG paths.')
else:
    display_cols = [
        'dataset', 'method', 'mode', 'samples', 'detected_anomalies',
        'precision', 'recall', 'f1', 'pr_auc'
    ]
    display(results_df[display_cols].sort_values(['dataset', 'f1'], ascending=[True, False]))

In [ ]:
if not results_df.empty:
    summary = (
        results_df
        .groupby('method', as_index=False)[['precision', 'recall', 'f1', 'pr_auc']]
        .mean()
        .sort_values('f1', ascending=False)
    )
    summary

## Optional: Export Results
Uncomment and run the next line to save results for your report.

`results_df.to_csv(project_root / 'report' / 'benchmark_results.csv', index=False)`

In [ ]:
results_df.to_csv(project_root / 'report' / 'benchmark_results.csv', index=False)